In [1]:
import os

In [2]:
%pwd

'd:\\DataScience\\Wine-Quality-MLOps-Pipeline\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\DataScience\\Wine-Quality-MLOps-Pipeline'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [9]:
class data_ingestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
import ssl
import urllib.request

# Global bypass for the SSL/TLS environment bug
ssl._create_default_https_context = ssl._create_unverified_context

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion_obj = data_ingestion(config=data_ingestion_config)
    data_ingestion_obj.download_file()
    data_ingestion_obj.extract_zip_file()
except Exception as e:
    raise e

[2026-07-20 23:08:37,995: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-07-20 23:08:38,000: INFO: common]: yaml file: params.yaml loaded successfully
[2026-07-20 23:08:38,001: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-07-20 23:08:38,003: INFO: common]: created directory at: artifacts
[2026-07-20 23:08:38,003: INFO: common]: created directory at: artifacts/data_ingestion


[2026-07-20 23:08:39,183: INFO: 3794568707]: artifacts/data_ingestion/data.zip downloaded with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: E6A0:2608CC:2C47CE:3D0CF8:6A5E5D15
Accept-Ranges: bytes
Date: Mon, 20 Jul 2026 17:38:30 GMT
Via: 1.1 varnish
X-Served-By: cache-del-vibw2260024-DEL
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1784569110.886076,VS0,VE393
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: a6e685416e6a72ba8a2119c63f673437b4c77b26
Expires: Mon, 20 Jul 2026 17:43:30 GMT
Source-Age: 0


